# The rocket digital twin: your notebook

This notebook runs **your** code: `twins.yaml` and `twin.py` in this folder. Use it next to your
editor: fill a gap in the code, come back, run the cells again.

| Time | Block | Given | You build |
|---|---|---|---|
| 14:30 | **Briefing A**: the physics | `physics.py` | |
| 14:30 | **Briefing B**: the Twin class | `Twin`, `TwinGraph.load`, `get` | |
| 14:45 | **Mission 1**: Structure | a spec sheet, one example relation | `twins.yaml`, `RELATIONS`, `validate()`, `out()`, `inc()` |
| 15:30 | **Mission 2**: State | your graph | `config()`, the go/no-go poll |
| 16:15 | **Briefing C**: the twin as a simulator | `fly()`, faults, the 5 % rules | |
| 16:25 | **Open challenge**: catch the failure | your twin, `CHALLENGE.md` | data, your own alert, data back into the twin, one screen |

`check()` or `check(1)`, defined in the first cell, runs `check.py` and shows what it says:
**PASS**, **TODO** or **FAIL**.

**Stuck?** The solutions open in a **terminal**, in this folder, with the password the instructor
gives at the end of each mission: `python unlock.py` lists them, `python unlock.py 1a` shows one,
`python unlock.py 1 --apply` puts all of Step 1 into `twin.py`.

Run the cells from top to bottom. Changes you make to `twin.py` are picked up automatically.

In [ ]:
%load_ext autoreload
%autoreload 2

import inspect
import math
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# The starter folder: the one with twin.py and check.py in it
ROOT = Path.cwd()
while not ((ROOT / "twin.py").exists() and (ROOT / "check.py").exists()):
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Open this notebook from the starter folder (the one with twin.py)")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import mission
import twin as tw                       # tw.RELATIONS: always your latest version
from mission import Reality, fly
from physics import G, Config, air_density, events, simulate
from twin import RocketTwin, Twin, TwinGraph

# How the charts look
plt.rcParams["figure.figsize"] = (10, 3.6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# The same four colours everywhere: plan, real, sensor, ok
PLAN, REAL, SENS, OK = "#2E5E9E", "#C0392B", "#D68910", "#1E8449"


def show(function):
    # Print the code of a function or class, so we can read it here.
    print(inspect.getsource(function))


def check(step=""):
    # Run check.py (every step, or only one) and show what it says.
    args = [sys.executable, "check.py"] + ([str(step)] if step else [])
    result = subprocess.run(args, cwd=ROOT, capture_output=True, text=True)
    print(result.stdout + result.stderr)


print("starter folder:", ROOT)

## What is a digital twin?

> A **virtual copy of one real thing**, kept in step with it by **live data**, used to **spot
> trouble** and to **predict** what will happen.

It is more than a dashboard: a dashboard shows data but has no idea what *should* happen. It is
more than a simulation: a simulation has no link to the real thing. A twin needs five
ingredients, and the afternoon builds them in order:

| Ingredient | Question | Where |
|---|---|---|
| **Model** | How should it behave? | Briefing A: the physics |
| **Structure** | What is it made of? | Briefing B and Mission 1: the twin graph |
| **State** | What condition is it in now? | Mission 2: `config()`, the poll |
| **Live data** | What do the sensors say? | Briefing C: the simulator `fly()` |
| **Comparison** | Does reality match the plan? | Briefing C and your challenge: alarms, your own alert |

---
## Briefing A · The physics (given)

`physics.py` is complete. You will not edit it: it is a function you call.

The rocket flies straight up. Every tenth of a second the model

1. adds up the forces: the engines push up, gravity pulls down, air resistance opposes the motion,
2. divides by the mass to get the acceleration (Newton's second law),
3. updates the speed and then the height,
4. burns some propellant, so the rocket gets lighter.

$$
a = \frac{T - m g - D}{m}, \qquad D = \tfrac12 \rho v^2 C_d A, \qquad \rho = 1.225\, e^{-h/8500}
$$

The model takes a `Config`: six numbers that describe the rocket. Here we type them by hand. In
Mission 2 your twin computes them from the graph.

In [ ]:
cfg_by_hand = Config(thrust_n=7_605_000, prop_kg=400_000, burn_kg_s=400_000 / 150,
                     dry_kg=28_200, cd=0.3, area_m2=math.pi * 1.85**2)
plan = simulate(cfg_by_hand)
print(f"{len(plan):,} time steps of 0.1 s")
plan.head()

The output is a **pandas DataFrame**, one row per time step: the **plan**, how the rocket *should*
fly. Flight events are just **queries** on it:

In [ ]:
ev = events(plan)
print(f"Max-Q (peak air pressure on the way up): T+{ev['max_q']:.0f} s")
print(f"MECO (engines off, tanks empty):         T+{ev['meco']:.0f} s")
print(f"Apogee (highest point):                  T+{ev['apogee']:.0f} s at {ev['apogee_h'] / 1000:,.0f} km")
print(f"Splashdown:                              T+{ev['end']:.0f} s")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))

axes[0].plot(plan.t, plan.h / 1000, color=PLAN)
axes[0].set_title("altitude [km]")
axes[1].plot(plan.t, plan.v / 1000, color=PLAN)
axes[1].set_title("velocity [km/s]")
ascent = plan[plan.t <= 150]
axes[2].plot(ascent.t, ascent.q / 1000, color=PLAN)
axes[2].set_title("air pressure on the way up [kPa]")

for ax in axes:
    ax.axvline(ev["meco"], color="gray", ls=":")       # a dotted line when the engines stop
    ax.set_xlabel("t [s]")

**A small puzzle.** Why does the air pressure peak at about T+46 s while the rocket is still speeding
up? Pressure is density times speed squared. Plot both, scaled to their maximum:

In [ ]:
first_2_min = plan[plan.t <= 120].set_index("t")
factors = pd.DataFrame()
factors["air density"] = first_2_min.h.apply(air_density)
factors["speed squared"] = first_2_min.v ** 2
factors["their product (air pressure)"] = first_2_min.q
(factors / factors.max()).plot(color=[OK, SENS, PLAN], ylabel="scaled to 1",
                               title="Max-Q: one factor rises, the other falls");

---
## Briefing B · The Twin class (given)

A **twin** is one *particular* thing: **this** engine, `eng-3`, mounted on **this** stage. The
`Twin` class is a record with six fields:

In [ ]:
show(Twin)

| Field | Meaning | Example |
|---|---|---|
| `id` | unique key, used by links | `"rocket"` |
| `type` | what kind of thing it is | `"Rocket"` |
| `name` | a human name | `"Aether-2"` |
| `props` | **design values** that do not change | `{"diameter_m": 3.7, ...}` |
| `state` | **values that change** during operations | an engine's `status`, a tank's level |
| `links` | **relations**: relation name → list of ids | `{"HAS_STAGE": ["stage-1", "stage-2"], ...}` |

One entry in `twins.yaml` becomes one `Twin`. The rocket is written for you as an example:

In [ ]:
graph = TwinGraph.load()          # read twins.yaml
rocket = graph.get("rocket")      # one twin
vars(rocket)                      # all its fields, as a dict

`TwinGraph` holds every twin in a dictionary, keyed by `id`:

| Method | What it does | Status |
|---|---|---|
| `TwinGraph.load()` | reads `twins.yaml`, builds one `Twin` per entry, validates, refuses a bad file | given |
| `get(id)`, `of_type(type)` | one twin by id, all twins of a type | given |
| `validate()` | checks every link against `RELATIONS`, returns a list of problems | **Mission 1** |
| `out()`, `inc()` | follow a link forwards or backwards | **Mission 1** |

In [ ]:
show(TwinGraph.load)

---
## Mission 1 · Structure: the rocket as data

| | |
|---|---|
| **When** | 14:45–15:30 (45 min) |
| **Given** | the `Twin` class and loading; a spec sheet at the top of `twins.yaml`; one example relation, `HAS_STAGE` |
| **You build** | every part in `twins.yaml`; the rest of `RELATIONS` (`TODO(1a)`); `validate()` (`1b`); `out()` (`1c`) and `inc()` (`1d`) |
| **Gate** | `check(1)` → PASS |
| **Solution unlocks** | 15:30, in a terminal: `python unlock.py 1 --apply` |

In [ ]:
check(1)

### Your twins as tables

Re-run these cells after every change to `twins.yaml`. One row per twin, then one row per link:

In [ ]:
graph = TwinGraph.load()

rows = []
for t in graph.twins.values():
    rows.append({"id": t.id, "type": t.type, "name": t.name, "props": t.props, "state": t.state})
twins = pd.DataFrame(rows)
twins

In [ ]:
rows = []
for t in graph.twins.values():
    for relation, targets in t.links.items():
        for target in targets:
            rows.append({"from": t.id, "relation": relation, "to": target})
edges = pd.DataFrame(rows, columns=["from", "relation", "to"])

print(len(twins), "twins,", len(edges), "links  (the spec sheet has 19 twins)")
edges.relation.value_counts()

### Seeing your graph

`networkx` draws it. A **red** arrow breaks your `RELATIONS` rules, or points to an id that does not
exist (it then shows up as a box of its own).

In [ ]:
import networkx as nx


def to_networkx(graph):
    # One node per twin, one edge per link. ok=False if the link breaks the RELATIONS rules.
    G = nx.DiGraph()
    for t in graph.twins.values():
        G.add_node(t.id)
    for t in graph.twins.values():
        for relation, targets in t.links.items():
            for target in targets:
                target_type = graph.twins[target].type if target in graph.twins else None
                ok = tw.RELATIONS.get(relation) == (t.type, target_type)
                if G.has_edge(t.id, target) and not G.edges[t.id, target]["ok"]:
                    continue                           # two links between the same twins: keep the bad one
                G.add_edge(t.id, target, relation=relation, ok=ok)
    return G


def draw_graph(G, title, figsize=(14, 6)):
    # The row of each node = how many links it is away from the first node (the rocket)
    first = list(G.nodes)[0]
    distance = nx.shortest_path_length(G.to_undirected(), first)
    far = max(distance.values()) + 1                   # nodes not connected: one row below the rest
    rows = {}
    for node in G.nodes:
        rows.setdefault(-distance.get(node, far), []).append(node)
    pos = nx.multipartite_layout(G, subset_key=rows, align="horizontal")

    colors = ["gray" if data["ok"] else "red" for a, b, data in G.edges(data=True)]
    labels = {}
    for a, b, data in G.edges(data=True):              # each relation labelled once (and every red one)
        if data["relation"] not in labels.values() or not data["ok"]:
            labels[(a, b)] = data["relation"]

    plt.figure(figsize=figsize)
    nx.draw_networkx_edges(G, pos, edge_color=colors, node_size=1500, arrowsize=12)
    nx.draw_networkx_edge_labels(G, pos, edge_labels=labels, font_size=7)
    nx.draw_networkx_labels(G, pos, font_size=8, bbox=dict(boxstyle="round", fc="white", ec="gray"))
    plt.title(title, loc="left")
    plt.axis("off")


draw_graph(to_networkx(graph), "Your twin graph, as loaded from twins.yaml")

### Your ontology: `RELATIONS` (`TODO(1a)`)

For each relation: which type of twin it starts from, and which type it points to. You start with
`HAS_STAGE`. Every relation used in your YAML needs a rule here:

In [ ]:
used = set(edges.relation)
rules = pd.DataFrame(tw.RELATIONS, index=["from type", "to type"]).T
print("used in twins.yaml but no rule yet:", sorted(used - set(rules.index)) or "none")
rules

### `validate()` catches mistakes (`TODO(1b)`)

We make a mistake on purpose, on a **copy** of the graph: the LOX tank listed among the engines of
stage 1. Your `validate()` should report it. (Until you write it, it returns `[]`: everything passes.)

In [ ]:
broken = graph.copy()
if "stage-1" in broken.twins:
    broken.get("stage-1").links.setdefault("HAS_ENGINE", []).append("tank-lox")
    print(broken.validate())
else:
    print("Add stage-1 to twins.yaml first.")

### Two queries do almost everything (`TODO(1c)`, `TODO(1d)`)

`out()` follows a relation forwards, `inc()` backwards. Expected: 9 engines, 2 tanks, 2 sensors.

In [ ]:
try:
    stage1 = graph.get("stage-1")
    print("engines of stage 1:  ", [e.name for e in graph.out(stage1, "HAS_ENGINE")])
    print("tanks of stage 1:    ", [t.name for t in graph.out(stage1, "HAS_TANK")])
    print("sensors watching it: ", [s.name for s in graph.inc(stage1, "MONITORS")])
except NotImplementedError as todo:
    print("Not done yet:", todo)
except KeyError as missing:
    print("Not in twins.yaml yet:", missing)

**Stuck?** At 15:30 the instructor gives the password. In a terminal, in this folder:

```bash
python unlock.py 1a                   # see the solution of one gap
python unlock.py 1 --apply            # put every Step 1 solution into twin.py
python unlock.py twins.yaml --apply   # the full twins.yaml (yours is kept as twins.yaml.mine)
```

Then come back and run the cells again.

---
## Mission 2 · State: from graph to numbers, and the go/no-go poll

| | |
|---|---|
| **When** | 15:30–16:05 (35 min) |
| **Given** | your graph from Mission 1 (or the unlocked one); the physics |
| **You build** | `RocketTwin.config()` (`TODO(3a)`); the poll checks in `go_no_go()` (`3b`); a what-if |
| **Gate** | `check()` → steps 1–4 PASS, and you can say what one failed engine does |
| **Solution unlocks** | 16:05, in a terminal: `python unlock.py 3 --apply` |

### Graph to numbers: `config()` (`TODO(3a)`)

In Briefing A we typed the six numbers by hand. Your twin should compute the same numbers by walking
the graph. Watch the units: the YAML says kN, the physics wants N.

In [ ]:
try:
    graph = TwinGraph.load()
    twin = RocketTwin(graph)
    cfg = twin.config()
    display(pd.DataFrame({"typed by hand": vars(cfg_by_hand), "from your graph": vars(cfg)}))
except NotImplementedError as todo:
    print("Not done yet:", todo)

### The go/no-go poll (`TODO(3b)`)

Expected with the spec sheet: five checks, all ok.

In [ ]:
try:
    display(pd.DataFrame(RocketTwin(TwinGraph.load()).go_no_go()))
except NotImplementedError as todo:
    print("Not done yet:", todo)

### What-if: change the data, not the code

Mark engines as failed in the data and ask the twin what happens:

In [ ]:
try:
    rows = []
    for n_failed in range(5):                          # 0, 1, 2, 3, 4 failed engines
        g = TwinGraph.load()
        for i in range(1, n_failed + 1):
            g.get(f"eng-{i}").state["status"] = "failed"
        what_if = RocketTwin(g)
        c = what_if.config()
        thrust_to_weight = c.thrust_n / ((c.dry_kg + c.prop_kg) * G)
        apogee_km = round(events(simulate(c))["apogee_h"] / 1000) if thrust_to_weight > 1 else 0
        rows.append({"failed engines": n_failed, "thrust [kN]": round(c.thrust_n / 1000),
                     "thrust / weight": round(thrust_to_weight, 2), "go?": what_if.ready(),
                     "planned apogee [km]": apogee_km})
    display(pd.DataFrame(rows))
except NotImplementedError as todo:
    print("Not done yet:", todo)

**Side quests:** plot the planned apogee against the payload mass (0–500 t): at what mass does the
poll say NO GO? Which single YAML value changes the apogee most?

In [ ]:
check()

**Stuck?** At 16:05, in a terminal: `python unlock.py 3 --apply` puts `config()` and `go_no_go()`
into `twin.py`.

---
## Briefing C · The twin as a simulator (given)

`mission.fly()` flies a "real" rocket. Reality runs the same physics, but with **faults the twin is
not told about**. The twin only sees its sensors, an IMU and an altimeter, and compares them with the
plan. This needs Missions 1 and 2 (yours, or the unlocked ones).

| Fault | What really happens |
|---|---|
| `thrust` | the engines drop to 70 % thrust |
| `leak` | propellant drains twice as fast |
| `freeze` | the altimeter stops updating |

In [ ]:
try:
    twin = RocketTwin(TwinGraph.load())
    twin.config()
except NotImplementedError as todo:
    raise RuntimeError(f"Briefing C needs Missions 1 and 2 first: {todo} is not done yet. "
                       "Finish them, or unlock them in a terminal (python unlock.py).") from None

scenarios = {"healthy": {}, "frozen altimeter at T+60": {"freeze": 60},
             "thrust loss at T+40": {"thrust": 40}, "leak at T+40": {"leak": 40}}
flights = {name: fly(twin, faults) for name, faults in scenarios.items()}

fig, axes = plt.subplots(1, 4, figsize=(15, 3.3), sharey=True)
for ax, name in zip(axes, flights):
    df = flights[name].df
    first_300s = df[df.t <= 300]
    ax.plot(first_300s.t, first_300s.plan_h / 1000, "--", color=PLAN, label="plan")
    ax.plot(first_300s.t, first_300s.h / 1000, "-", color=REAL, label="real (true)")
    ax.plot(first_300s.t, first_300s.altimeter / 1000, ":", color=SENS, label="altimeter")
    ax.set_title(name)
    for kind, t_alarm in flights[name].alarms.items():
        ax.axvline(t_alarm, color="k", lw=0.8)
        ax.text(t_alarm + 4, 20, f"{kind} alarm", fontsize=8)
axes[0].set_ylabel("altitude [km]")
axes[0].legend();

In [ ]:
rows = []
for name, f in flights.items():
    rows.append({"scenario": name,
                 "sensor alarm": f.alarms.get("sensor"),
                 "vehicle alarm": f.alarms.get("vehicle"),
                 "diagnosis": f.diagnosis,
                 "predicted apogee [km]": None if f.replanned_apogee is None else round(f.replanned_apogee / 1000),
                 "real apogee [km]": round(f.apogee / 1000)})
pd.DataFrame(rows)

### Two kinds of alarm: the 5 % rules, your baseline

In [ ]:
show(mission.sensor_alarm)
show(mission.vehicle_alarm)

- **Sensor vs sensor** catches the frozen altimeter within about a second.
- **Sensor vs plan** catches the thrust loss and the leak, but only after the rocket is 5 % off.

> When the *rocket* is wrong, every sensor agrees, and agrees on the wrong thing. You need an
> **expected value**, and that is what the plan gives you.

### Diagnosis by fingerprint

In [ ]:
thrust = flights["thrust loss at T+40"].df.query("t <= 160")
leak = flights["leak at T+40"].df.query("t <= 160")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
ax[0].plot(thrust.t, thrust.imu_a, color=REAL, label="thrust loss")
ax[0].plot(leak.t, leak.imu_a, color=SENS, label="leak")
ax[0].plot(leak.t, leak.plan_a, "--", color=PLAN, label="plan")
ax[0].set_title("acceleration [m/s²]")
ax[0].legend()
ax[1].plot(thrust.t, thrust.tank_t, color=REAL)
ax[1].plot(leak.t, leak.tank_t, color=SENS)
ax[1].plot(leak.t, leak.plan_fuel_t, "--", color=PLAN)
ax[1].set_title("propellant left [t]");

In [ ]:
show(mission.diagnose)
pd.DataFrame(flights["thrust loss at T+40"].log)[["t", "title", "text"]]

### No two rockets are alike

`Reality.random(seed)` gives each flight a small **hidden thrust delta** (about ±0.5 %, at most
±2 %) and noisy sensors. The plan still uses the rated thrust. Twenty healthy flights:

In [ ]:
t_apogee = ev["apogee"]
plt.figure(figsize=(10, 3.6))
for seed in range(20):
    reality = Reality.random(seed)
    f = fly(twin, reality=reality, t_max=t_apogee)
    d = f.df[f.df.t > 20]
    percent_off = (d.imu_h - d.plan_h) / d.plan_h * 100
    plt.plot(d.t, percent_off, lw=1, color=REAL if reality.thrust_factor > 1 else PLAN, alpha=0.7)
plt.axhspan(5, 6, color=REAL, alpha=0.1)
plt.axhspan(-6, -5, color=REAL, alpha=0.1)
plt.ylim(-6, 6)
plt.xlabel("t [s]")
plt.ylabel("measured minus plan [%]")
plt.title("20 healthy flights: red = stronger engines than rated, blue = weaker; shaded = alarm zone");

Every healthy flight drifts a little, but stays inside the 5 % alarm. **Where do you draw the line
between normal variation and a fault?** That is your challenge.

---
## Open challenge · Catch the failure (16:25–17:40)

> **Can your twin learn what a normal flight looks like, and catch a failure earlier than the 5 % rules?**

No `TODO` and no `check.py` from here: you choose the method and the tools. `CHALLENGE.md` has the
details. **The minimum** (all four are needed for the demo):

1. **Data.** At least 100 simulated flights, healthy and faulty, each with its own
   `Reality.random(seed)`, labelled (which fault, at what time), saved to a file.
2. **Your own alert.** Report the false-alarm rate on healthy flights, the miss rate for each fault
   type, and the detection delay next to the delay of the 5 % rules (`flight.alarms`).
3. **Data back into the twin.** Record flights and alerts in the twin graph; `validate()` must
   still pass. Then answer with a query: *which flights raised an alert?*
4. **One screen** that shows a faulty flight: plan against what the sensors saw, and the moment your
   alert fires.

| Columns of `f.df` | |
|---|---|
| what the twin **sees**: use these | `altimeter`, `imu_h`, `imu_v`, `imu_a`, `tank_t` |
| what the twin **expects**: use these | `plan_h`, `plan_v`, `plan_a`, `plan_fuel_t` |
| the hidden **truth**: do NOT use | `h`, `v`, `a`, `q`, `fuel_t`, `thrust_kn`, `f.reality` |

**A start for the data:**

In [ ]:
tables = []
for seed in range(3):
    f = fly(twin, {"thrust": 40}, Reality.random(seed), t_max=200)
    df = f.df[f.df.t % 1 == 0].copy()      # one row per second is enough
    df["run_id"] = seed
    df["fault"] = "thrust"
    df["t_fault"] = 40
    tables.append(df)
data = pd.concat(tables)
print(data.shape)
data.head()

### Your work

**After the demos**, one reference answer unlocks at 17:40. It is one way among many, saved to
`revealed/` so it never overwrites your work. In a terminal: `python unlock.py experiments.py`,
`python unlock.py anomaly.py`, `python unlock.py app.py`.

---
## Five things to remember

1. A digital twin = **structure** + **model** + **state** + **live data** + **comparison**.
2. Structure lives in **data**, not in code. Change the YAML and the twin follows.
3. Compare reality with the **plan**, not only sensor with sensor: many faults fool every sensor at once.
4. A twin **predicts**: from the current state it tells you where you will end up.
5. The plan gives every reading an **expected value**: residuals from it make excellent features, and
   simulated flights give you the labelled data to learn from.